In [1]:
"""
01 — carOBD Data Exploration
Goal of this notebook:
  1. See what the carOBD CSVs actually look like (columns, sample rate, scale)
  2. Confirm the 6 'signature PIDs' our fault taxonomy needs are present and usable
  3. Compute healthy baseline stats (mean / std / min / max per PID)
  4. Spot any data quality issues that would break Week 2's injection engine

This notebook eventually feeds docs/DATA_NOTES.md.
"""
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Repo-relative paths. Notebooks live in notebooks/, so '..' gets us to the repo root.
REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "raw" / "carOBD"

# Sanity check: does the data dir exist?
assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"
print(f"Data dir: {DATA_DIR}")
print(f"Files found: {len(list(DATA_DIR.glob('*.csv')))}")

Data dir: D:\Predictive-MaintenanceV1\Predictive-MaintenanceV1\data\raw\carOBD
Files found: 129


In [2]:
# Group files by their prefix (idle/drive/live/ufpe/long)
# This matches the README's file-naming convention.
csv_files = sorted(DATA_DIR.glob("*.csv"))

from collections import defaultdict
files_by_kind = defaultdict(list)
for f in csv_files:
    # Pull off trailing digits to get the 'kind' (e.g. drive3.csv -> drive)
    kind = "".join(c for c in f.stem if not c.isdigit())
    files_by_kind[kind].append(f.name)

for kind, names in sorted(files_by_kind.items()):
    print(f"  {kind:8s} : {len(names):2d} files  ->  {names[:3]}{'...' if len(names) > 3 else ''}")

  drive    : 13 files  ->  ['drive1.csv', 'drive10.csv', 'drive11.csv']...
  idle     : 47 files  ->  ['idle1.csv', 'idle10.csv', 'idle11.csv']...
  live     : 39 files  ->  ['live1.csv', 'live10.csv', 'live11.csv']...
  long     : 12 files  ->  ['long1.csv', 'long10.csv', 'long11.csv']...
  ufpe     : 18 files  ->  ['ufpe1.csv', 'ufpe10.csv', 'ufpe11.csv']...


In [3]:
# Pick a 'drive' file as our first specimen because it should have the richest signal
# (idle files might have all PIDs constant; drive files exercise the engine across regimes).
sample_path = DATA_DIR / "drive1.csv"
df = pd.read_csv(sample_path)

print(f"File: {sample_path.name}")
print(f"Shape: {df.shape}  (rows = seconds at 1 Hz)")
print(f"Duration: {len(df)} s  ≈  {len(df)/60:.1f} min")
print()
print("Columns:")
for c in df.columns:
    print(f"  {c}")

File: drive1.csv
Shape: (2709, 27)  (rows = seconds at 1 Hz)
Duration: 2709 s  ≈  45.1 min

Columns:
  ENGINE_RUN_TINE ()
  ENGINE_RPM ()
  VEHICLE_SPEED ()
  THROTTLE ()
  ENGINE_LOAD ()
  COOLANT_TEMPERATURE ()
  LONG_TERM_FUEL_TRIM_BANK_1 ()
  SHORT_TERM_FUEL_TRIM_BANK_1 ()
  INTAKE_MANIFOLD_PRESSURE ()
  FUEL_TANK ()
  ABSOLUTE_THROTTLE_B ()
  PEDAL_D ()
  PEDAL_E ()
  COMMANDED_THROTTLE_ACTUATOR ()
  FUEL_AIR_COMMANDED_EQUIV_RATIO ()
  ABSOLUTE_BAROMETRIC_PRESSURE ()
  RELATIVE_THROTTLE_POSITION ()
  INTAKE_AIR_TEMP ()
  TIMING_ADVANCE ()
  CATALYST_TEMPERATURE_BANK1_SENSOR1 ()
  CATALYST_TEMPERATURE_BANK1_SENSOR2 ()
  CONTROL_MODULE_VOLTAGE ()
  COMMANDED_EVAPORATIVE_PURGE ()
  TIME_RUN_WITH_MIL_ON ()
  TIME_SINCE_TROUBLE_CODES_CLEARED ()
  DISTANCE_TRAVELED_WITH_MIL_ON ()
  WARM_UPS_SINCE_CODES_CLEARED ()


In [4]:
print("First 3 rows:")
display(df.head(3))
print("\nLast 3 rows:")
display(df.tail(3))


First 3 rows:


,ENGINE_RUN_TINE (),ENGINE_RPM (),VEHICLE_SPEED (),THROTTLE (),ENGINE_LOAD (),COOLANT_TEMPERATURE (),LONG_TERM_FUEL_TRIM_BANK_1 (),SHORT_TERM_FUEL_TRIM_BANK_1 (),INTAKE_MANIFOLD_PRESSURE (),FUEL_TANK (),...,INTAKE_AIR_TEMP (),TIMING_ADVANCE (),CATALYST_TEMPERATURE_BANK1_SENSOR1 (),CATALYST_TEMPERATURE_BANK1_SENSOR2 (),CONTROL_MODULE_VOLTAGE (),COMMANDED_EVAPORATIVE_PURGE (),TIME_RUN_WITH_MIL_ON (),TIME_SINCE_TROUBLE_CODES_CLEARED (),DISTANCE_TRAVELED_WITH_MIL_ON (),WARM_UPS_SINCE_CODES_CLEARED ()
0,0,0.0,0,17.647058,0.0,81,-4.6875,0.0,101,60.392159,...,45,5,424.799988,251.5,12.382,0.0,0,8260,0,255
1,0,0.0,0,17.647058,0.0,81,-4.6875,0.0,101,60.392159,...,45,5,424.799988,251.5,12.382,0.0,0,8260,0,255
2,0,0.0,0,17.647058,0.0,81,-4.6875,0.0,101,60.392159,...,45,5,424.799988,251.5,12.363,0.0,0,8260,0,255



Last 3 rows:


,ENGINE_RUN_TINE (),ENGINE_RPM (),VEHICLE_SPEED (),THROTTLE (),ENGINE_LOAD (),COOLANT_TEMPERATURE (),LONG_TERM_FUEL_TRIM_BANK_1 (),SHORT_TERM_FUEL_TRIM_BANK_1 (),INTAKE_MANIFOLD_PRESSURE (),FUEL_TANK (),...,INTAKE_AIR_TEMP (),TIMING_ADVANCE (),CATALYST_TEMPERATURE_BANK1_SENSOR1 (),CATALYST_TEMPERATURE_BANK1_SENSOR2 (),CONTROL_MODULE_VOLTAGE (),COMMANDED_EVAPORATIVE_PURGE (),TIME_RUN_WITH_MIL_ON (),TIME_SINCE_TROUBLE_CODES_CLEARED (),DISTANCE_TRAVELED_WITH_MIL_ON (),WARM_UPS_SINCE_CODES_CLEARED ()
2706,1330,658.5,0,15.686275,23.921568,89,-1.5625,-2.34375,24,56.07843,...,56,1,471.299988,351.799988,13.691,22.352942,0,8282,0,255
2707,1330,658.5,0,15.686275,23.921568,89,-1.5625,-2.34375,24,56.07843,...,56,1,470.700012,350.700012,13.710,21.960785,0,8282,0,255
2708,1330,658.5,0,15.686275,23.921568,89,-1.5625,-2.34375,24,56.07843,...,56,1,470.700012,350.700012,13.710,21.960785,0,8282,0,255


In [3]:
"""
Look at actual rows across files that the audit flagged differently.

Goal: figure out whether the 'suspicious' files have a different units/encoding,
or whether they're really capturing different physical conditions.
"""
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "raw" / "carOBD"

# 4 files that should give us a representative cross-section.
# - drive1: looked clean in the audit (STFT in [-7.8, 11.7])
# - drive10: looked suspicious (STFT in [19, 101])
# - live5: looked clean (STFT in [-9.4, 10.2])
# - idle1: looked suspicious (STFT in [23, 101])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 30)
files = ["drive1.csv", "drive10.csv", "live5.csv", "idle1.csv"]

cols = [
    "ENGINE_RPM ()",
    "VEHICLE_SPEED ()",
    "COOLANT_TEMPERATURE ()",
    "SHORT_TERM_FUEL_TRIM_BANK_1 ()",
    "LONG_TERM_FUEL_TRIM_BANK_1 ()",
    "TIMING_ADVANCE ()",
    "INTAKE_MANIFOLD_PRESSURE ()",
    "ENGINE_LOAD ()",
]

for f in files:
    path = DATA_DIR / f
    if not path.exists():
        print(f"\n!! {f} not found at {path}")
        continue
    df = pd.read_csv(path)
    cols_present = [c for c in cols if c in df.columns]
    mid = len(df) // 2

    print(f"\n{'='*70}")
    print(f"{f}  ({len(df)} rows)")
    print(f"{'='*70}")
    print("First 3 rows:")
    print(df[cols_present].head(3).to_string())
    print("\nMiddle 3 rows:")
    print(df[cols_present].iloc[mid:mid+3].to_string())
    print("\nLast 3 rows:")
    print(df[cols_present].tail(3).to_string())


drive1.csv  (2709 rows)
First 3 rows:
   ENGINE_RPM ()  VEHICLE_SPEED ()  COOLANT_TEMPERATURE ()  SHORT_TERM_FUEL_TRIM_BANK_1 ()  LONG_TERM_FUEL_TRIM_BANK_1 ()  TIMING_ADVANCE ()  INTAKE_MANIFOLD_PRESSURE ()  ENGINE_LOAD ()
0            0.0                 0                      81                             0.0                        -4.6875                  5                          101             0.0
1            0.0                 0                      81                             0.0                        -4.6875                  5                          101             0.0
2            0.0                 0                      81                             0.0                        -4.6875                  5                          101             0.0

Middle 3 rows:
      ENGINE_RPM ()  VEHICLE_SPEED ()  COOLANT_TEMPERATURE ()  SHORT_TERM_FUEL_TRIM_BANK_1 ()  LONG_TERM_FUEL_TRIM_BANK_1 ()  TIMING_ADVANCE ()  INTAKE_MANIFOLD_PRESSURE ()  ENGINE_LOAD ()
1354        

In [2]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "raw" / "carOBD"

# Get the column list from each file and group files by their header
schemas = {}
for path in sorted(DATA_DIR.glob("*.csv")):
    cols = tuple(pd.read_csv(path, nrows=0).columns)  # nrows=0 reads only the header
    schemas.setdefault(cols, []).append(path.name)

print(f"Found {len(schemas)} distinct schemas across {sum(len(v) for v in schemas.values())} files\n")
for i, (cols, files) in enumerate(schemas.items(), 1):
    print(f"--- Schema {i} ({len(cols)} columns, {len(files)} files) ---")
    print(f"   Files: {files[:5]}{'...' if len(files) > 5 else ''}")
    print(f"   Columns: {list(cols)}")
    print()

Found 1 distinct schemas across 129 files

--- Schema 1 (27 columns, 129 files) ---
   Files: ['drive1.csv', 'drive10.csv', 'drive11.csv', 'drive12.csv', 'drive13.csv']...
   Columns: ['ENGINE_RUN_TINE ()', 'ENGINE_RPM ()', 'VEHICLE_SPEED ()', 'THROTTLE ()', 'ENGINE_LOAD ()', 'COOLANT_TEMPERATURE ()', 'LONG_TERM_FUEL_TRIM_BANK_1 ()', 'SHORT_TERM_FUEL_TRIM_BANK_1 ()', 'INTAKE_MANIFOLD_PRESSURE ()', 'FUEL_TANK ()', 'ABSOLUTE_THROTTLE_B ()', 'PEDAL_D ()', 'PEDAL_E ()', 'COMMANDED_THROTTLE_ACTUATOR ()', 'FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'ABSOLUTE_BAROMETRIC_PRESSURE ()', 'RELATIVE_THROTTLE_POSITION ()', 'INTAKE_AIR_TEMP ()', 'TIMING_ADVANCE ()', 'CATALYST_TEMPERATURE_BANK1_SENSOR1 ()', 'CATALYST_TEMPERATURE_BANK1_SENSOR2 ()', 'CONTROL_MODULE_VOLTAGE ()', 'COMMANDED_EVAPORATIVE_PURGE ()', 'TIME_RUN_WITH_MIL_ON ()', 'TIME_SINCE_TROUBLE_CODES_CLEARED ()', 'DISTANCE_TRAVELED_WITH_MIL_ON ()', 'WARM_UPS_SINCE_CODES_CLEARED ()']



In [4]:
"""
Physical bounds check.

For every file, look at the values in 4 columns and check whether they fall
within the physically possible range for that PID. If a column violates its
bounds, the column header doesn't match what's actually in the column.
"""
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "raw" / "carOBD"

# (column_name, low, high, description)
# Bounds are deliberately generous — anything outside means the column
# is NOT actually that PID.
BOUNDS = [
    ("VEHICLE_SPEED ()",            0,    200, "0-200 km/h"),
    ("COOLANT_TEMPERATURE ()",      -10,  130, "-10 to 130 C"),
    ("TIMING_ADVANCE ()",           -64,  64,  "-64 to 64 deg"),
    ("SHORT_TERM_FUEL_TRIM_BANK_1 ()", -100, 100, "-100 to 100 %"),
]

rows = []
for path in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(path)
    row = {"file": path.name}
    for col, lo, hi, _ in BOUNDS:
        s = df[col].dropna()
        if len(s) == 0:
            row[col] = "EMPTY"
        elif s.min() >= lo and s.max() <= hi:
            row[col] = "OK"
        else:
            row[col] = f"OUT [{s.min():.1f}, {s.max():.1f}]"
    rows.append(row)

result = pd.DataFrame(rows)

# Group files by their pattern across the 4 checks
result["pattern"] = result.apply(
    lambda r: "|".join("OK" if "OK" == r[c] else "BAD"
                       for c in [b[0] for b in BOUNDS]),
    axis=1,
)
print("Group sizes by pass/fail pattern:")
print(result["pattern"].value_counts().to_string())
print()
print("Per-file detail (truncated):")
# Show a sample of each pattern
for pattern, group in result.groupby("pattern"):
    print(f"\n{'-'*70}\nPattern: {pattern}  ({len(group)} files)")
    print(group.head(8).to_string(index=False))

Group sizes by pass/fail pattern:
pattern
OK|OK|BAD|BAD    108
OK|OK|BAD|OK      12
OK|OK|OK|OK        9

Per-file detail (truncated):

----------------------------------------------------------------------
Pattern: OK|OK|BAD|BAD  (108 files)
       file VEHICLE_SPEED () COOLANT_TEMPERATURE () TIMING_ADVANCE () SHORT_TERM_FUEL_TRIM_BANK_1 ()       pattern
drive10.csv               OK                     OK OUT [48.0, 629.6]              OUT [19.0, 101.0] OK|OK|BAD|BAD
drive11.csv               OK                     OK OUT [62.1, 609.1]              OUT [19.0, 101.0] OK|OK|BAD|BAD
drive12.csv               OK                     OK OUT [30.5, 533.3]              OUT [18.0, 101.0] OK|OK|BAD|BAD
drive13.csv               OK                     OK OUT [45.3, 621.9]              OUT [18.0, 101.0] OK|OK|BAD|BAD
 drive2.csv               OK                     OK OUT [61.9, 506.4]              OUT [19.0, 101.0] OK|OK|BAD|BAD
 drive3.csv               OK                     OK OUT [38.4, 594.

In [5]:
"""
How much usable data do the 9 clean files contain?
"""
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "raw" / "carOBD"

clean_files = [
    "drive1.csv",
    "live5.csv", "live6.csv", "live7.csv", "live8.csv",
    "live9.csv", "live10.csv", "live11.csv", "live12.csv",
]

total_rows = 0
print(f"{'file':<15}  {'rows':>6}  {'minutes':>8}")
print("-" * 35)
for f in clean_files:
    df = pd.read_csv(DATA_DIR / f)
    total_rows += len(df)
    print(f"{f:<15}  {len(df):>6}  {len(df)/60:>8.1f}")
print("-" * 35)
print(f"{'TOTAL':<15}  {total_rows:>6}  {total_rows/60:>8.1f}")
print(f"\n≈ {total_rows/3600:.2f} hours of healthy data at 1 Hz")

file               rows   minutes
-----------------------------------
drive1.csv         2709      45.1
live5.csv          2384      39.7
live6.csv          2287      38.1
live7.csv          2453      40.9
live8.csv          2413      40.2
live9.csv          2168      36.1
live10.csv         1176      19.6
live11.csv         1727      28.8
live12.csv         1151      19.2
-----------------------------------
TOTAL             18468     307.8

≈ 5.13 hours of healthy data at 1 Hz
